In [1]:
#| default_exp rest

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()

In [3]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

In [4]:
#| export
from rest.core import init_instance, generate
singleton, model_path = init_instance()

In [5]:
model_path = 'pelevin'

In [6]:
#| export
seq_length = 1024

model_path = f'./models/large/{model_path}'
from transformers import GPT2LMHeadModel,GPT2Tokenizer
tokenizer = GPT2Tokenizer.from_pretrained(model_path, pad_token_id = 50256)
model = GPT2LMHeadModel.from_pretrained(model_path).half()
model.config.pad_token_id = model.config.eos_token_id
model.cuda()
model.eval();

In [7]:
sum(p.numel() for p in model.parameters())

774030080

In [8]:
#| export
import threading
lock = threading.RLock()

def get_sample(prompt, length:int, num_samples:int, allow_linebreak:bool):
    with lock:
        return generate(model, tokenizer, seq_length, prompt, length, num_samples, allow_linebreak)

In [9]:
%%time
get_sample(' - ты кто?', 50, 4, False)

not setting adaptive thresholding
CPU times: user 8.69 s, sys: 683 ms, total: 9.37 s
Wall time: 1.35 s


[' Операцию помнишь? Как заснул? Если уж не совсем как пьяный, скажи хотя бы на правах дружбы. Напиши, что пережил, а? Просвети. И проси еще, чтоб автору твоему разрешили иногда дружить. А?',
 ' Зачем ты здесь? Ты ведь умер… Ты можешь показать мне, как это? Я так хочу. Только надо будет всё обмозговать… Ну как, обещаешь? Ну дай мне слово. Что ты на это скажешь? Что скажешь, а?',
 ' Сосед? Мы знакомы? Ну, а дальше? Скажем, ты сейчас дал мне по морде, чтоб я не мешал тебе смотреть телевизор. А потом, значит, дашь по физиономии мне, как только я усну. Как это ты будешь делать?',
 ' Господи, я даже толком не помню… Господи… Жизнь моя! Я ж тебя… Покойник! Вот как оно… А ты его… убить хотел? Смерть ему. Тебе – жизнь, и брату.']